# 21. Prepare Blinded Human Annotation Sheet from COMET Outputs

This notebook builds a **blinded human-annotation package** from the aligned COMET source-of-truth data.

## Goal
Create a single annotation package with:

- **50 total examples**
  - 25 from `idioms_test`
  - 25 from `wmt_test`
- **6 model outputs per example**
  - `baseline`
  - `idiom_only_v1`
  - `lora_r4_stage2`
  - `lora_r8_stage2`
  - `lora_r16_stage2`
  - `flan_t5_small_prompt_0shot`

This notebook is the *preparation* stage only. It:
- selects the examples
- validates full model coverage
- pivots to wide format
- blinds outputs per example
- exports the blinded CSV and the private mapping JSON

It does **not** analyze completed annotations. That belongs in a later notebook.


## 0. Mount Drive
Run this cell first in a fresh Colab runtime.

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Paths and configuration
Adjust these if your project lives in a different Drive location.

In [8]:
import json
import os
import random
from pathlib import Path

import pandas as pd
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/ds266_idiom_mt")
QUAL_PREDS_DIR = PROJECT_ROOT / "qual_preds"
COMET_EVAL_DIR = QUAL_PREDS_DIR / "comet_eval"

COMET_SENTENCE_SCORES_PATH = COMET_EVAL_DIR / "comet_sentence_scores.csv"
COMET_SENTENCE_SPREAD_PATH = COMET_EVAL_DIR / "comet_sentence_spread.csv"

# If True, require these exact counts after example selection.
REQUIRE_EXACT_COUNTS = True
TARGET_COUNTS = {
    "idioms_test": 25,
    "wmt_test": 25,
}

# Final model set for human annotation:
# 5 trained MT models + 1 FLAN baseline.
MODEL_ORDER = [
    "baseline",
    "idiom_only_v1",
    "lora_r4_stage2",
    "lora_r8_stage2",
    "lora_r16_stage2",
    "flan_t5_small_prompt_0shot",
]

# If a split already has exactly TARGET_COUNTS[group] examples, use all of them.
# Otherwise, select the highest-spread examples according to comet_sentence_spread.csv.
SELECTION_MODE = "all_if_exact_else_top_spread"  # alternatives: "top_spread", "random"

OUT_BASENAME = "preds_6models_annotation_v3"
OUT_BLINDED_CSV = QUAL_PREDS_DIR / f"{OUT_BASENAME}_blinded.csv"
OUT_MAPPING_JSON = QUAL_PREDS_DIR / f"{OUT_BASENAME}_mapping_PRIVATE.json"
OUT_UNBLINDED_PREVIEW_CSV = QUAL_PREDS_DIR / f"{OUT_BASENAME}_unblinded_preview.csv"
OUT_SELECTED_EXAMPLES_CSV = QUAL_PREDS_DIR / f"{OUT_BASENAME}_selected_examples.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("COMET_SENTENCE_SCORES_PATH exists:", COMET_SENTENCE_SCORES_PATH.exists())
print("COMET_SENTENCE_SPREAD_PATH exists:", COMET_SENTENCE_SPREAD_PATH.exists())
print("Output directory exists:", QUAL_PREDS_DIR.exists())


PROJECT_ROOT: /content/drive/MyDrive/ds266_idiom_mt
COMET_SENTENCE_SCORES_PATH exists: True
COMET_SENTENCE_SPREAD_PATH exists: True
Output directory exists: True


## 2. Load and validate COMET source-of-truth files
The annotation package should be built from the aligned COMET data, not from older pre-alignment notebook outputs.

In [9]:
scores_df = pd.read_csv(COMET_SENTENCE_SCORES_PATH)
spread_df = pd.read_csv(COMET_SENTENCE_SPREAD_PATH)

required_score_cols = {"src", "ref", "pred", "group", "model"}
missing_score_cols = required_score_cols - set(scores_df.columns)
if missing_score_cols:
    raise ValueError(f"Missing required columns in comet_sentence_scores.csv: {missing_score_cols}")

required_spread_cols = {"src", "ref", "group", "comet_spread"}
missing_spread_cols = required_spread_cols - set(spread_df.columns)
if missing_spread_cols:
    raise ValueError(f"Missing required columns in comet_sentence_spread.csv: {missing_spread_cols}")

# Rename to stable internal names so the rest of the notebook stays readable
scores_df = scores_df.rename(columns={
    "src": "source",
    "ref": "reference",
    "pred": "prediction",
})
spread_df = spread_df.rename(columns={
    "src": "source",
    "ref": "reference",
    "comet_spread": "comet_range",
})

print("Sentence scores shape:", scores_df.shape)
print("Spread shape:", spread_df.shape)
print("\nGroups in scores_df:", sorted(scores_df["group"].dropna().unique().tolist()))
print("Models in scores_df:", sorted(scores_df["model"].dropna().unique().tolist()))


Sentence scores shape: (400, 11)
Spread shape: (50, 14)

Groups in scores_df: ['idioms_test', 'wmt_test']
Models in scores_df: ['baseline', 'flan_t5_small_prompt_0shot', 'flan_t5_small_prompt_3shot', 'flan_t5_small_prompt_5shot', 'idiom_only_v1', 'lora_r16_stage2', 'lora_r4_stage2', 'lora_r8_stage2']


## 3. Select the canonical 50 examples
This section selects:
- 25 from `idioms_test`
- 25 from `wmt_test`

Default behavior:
- if a group already contains exactly the target count, use all examples
- otherwise use the highest-COMET-spread examples for that group

In [10]:
def select_examples_for_group(group_name: str, target_n: int) -> pd.DataFrame:
    group_spread = spread_df[spread_df["group"] == group_name].copy()

    if group_spread.empty:
        raise ValueError(f"No examples found in spread file for group={group_name}")

    if SELECTION_MODE == "all_if_exact_else_top_spread" and len(group_spread) == target_n:
        selected = group_spread.copy()
    elif SELECTION_MODE in {"all_if_exact_else_top_spread", "top_spread"}:
        selected = group_spread.sort_values("comet_range", ascending=False).head(target_n).copy()
    elif SELECTION_MODE == "random":
        selected = group_spread.sample(n=min(target_n, len(group_spread)), random_state=SEED).copy()
    else:
        raise ValueError(f"Unknown SELECTION_MODE: {SELECTION_MODE}")

    if REQUIRE_EXACT_COUNTS and len(selected) != target_n:
        raise ValueError(
            f"group={group_name} selected {len(selected)} examples, expected {target_n}. "
            "Either relax REQUIRE_EXACT_COUNTS or adjust the source data."
        )

    selected["selection_rank_within_group"] = range(1, len(selected) + 1)
    return selected

selected_parts = []
for grp, n in TARGET_COUNTS.items():
    selected_parts.append(select_examples_for_group(grp, n))

selected_examples = pd.concat(selected_parts, ignore_index=True)

# Stable example ids: numeric, matching the style of the prior annotation CSV.
selected_examples = selected_examples.sort_values(["group", "source", "reference"]).reset_index(drop=True)
selected_examples["example_id"] = range(len(selected_examples))

print("Selected examples shape:", selected_examples.shape)
display(selected_examples.head())
display(selected_examples["group"].value_counts())


Selected examples shape: (50, 16)


,source,reference,group,baseline,flan_t5_small_prompt_0shot,flan_t5_small_prompt_3shot,flan_t5_small_prompt_5shot,idiom_only_v1,lora_r16_stage2,lora_r4_stage2,lora_r8_stage2,comet_max,comet_min,comet_range,selection_rank_within_group,example_id
0,"""Regardless of their cause, inequalities are a...","""Unabhängig von ihrer Ursache sind Ungleichhei...",idioms_test,0.842750,0.290774,0.307880,0.317754,0.876986,0.827501,0.860689,0.827501,0.876986,0.290774,0.586211,4,0
1,A recent public relations disaster left the co...,Die jüngste Katastrophe im Bereich der Öffentl...,idioms_test,0.619298,0.299107,0.299107,0.299107,0.526990,0.602276,0.619298,0.602276,0.619298,0.299107,0.320191,21,1
2,"Bend the knee, peasant! Admit that I am your r...","Beuge das Knie, Bauer! Gib zu, dass ich dein r...",idioms_test,0.799448,0.347663,0.319701,0.282064,0.917172,0.799448,0.799448,0.799448,0.917172,0.282064,0.635108,2,2
3,Burning the candle at both ends is a recipe fo...,"Raubbau mit der Gesundheit zu treiben, ist ein...",idioms_test,0.548028,0.403939,0.403939,0.403939,0.548028,0.548028,0.548028,0.548028,0.548028,0.403939,0.144088,25,3
4,But before she could launch her charm offensiv...,Doch bevor sie ihre Charmeoffensive starten ko...,idioms_test,0.932659,0.252559,0.169645,0.169645,0.863792,0.932659,0.945956,0.932659,0.945956,0.169645,0.776311,1,4


,count
group,
idioms_test,25
wmt_test,25


## 4. Filter sentence-level scores to the selected examples and models
This is where we enforce that every selected example has one prediction for each chosen model.

In [11]:
selected_key_cols = ["group", "source", "reference"]
selected_key_df = selected_examples[selected_key_cols + ["example_id"]].drop_duplicates()

merged = scores_df.merge(selected_key_df, on=selected_key_cols, how="inner")
merged = merged[merged["model"].isin(MODEL_ORDER)].copy()

# Resolve duplicate example_id columns created by merge
if "example_id_y" in merged.columns:
    merged["example_id"] = merged["example_id_y"]
elif "example_id_x" in merged.columns:
    merged["example_id"] = merged["example_id_x"]

merged = merged.drop(columns=[c for c in ["example_id_x", "example_id_y"] if c in merged.columns])

# Validate coverage: each example should have one row per model
coverage = (
    merged.groupby(["example_id", "group"])["model"]
    .nunique()
    .reset_index(name="n_models_present")
)

print("Expected number of models per example:", len(MODEL_ORDER))
print("Coverage summary:")
display(coverage["n_models_present"].value_counts().sort_index())

bad_examples = coverage[coverage["n_models_present"] != len(MODEL_ORDER)]
if not bad_examples.empty:
    display(bad_examples.head(20))
    raise ValueError(
        "Some selected examples are missing one or more models. "
        "Adjust MODEL_ORDER or selection criteria before continuing."
    )

# Also verify no duplicate model outputs per example
dup_check = merged.groupby(["example_id", "model"]).size().reset_index(name="n_rows")
dup_bad = dup_check[dup_check["n_rows"] != 1]
if not dup_bad.empty:
    display(dup_bad.head(20))
    raise ValueError("Found duplicate rows for at least one (example_id, model) pair.")

print("Filtered merged shape:", merged.shape)
display(merged.head())


Expected number of models per example: 6
Coverage summary:


,count
n_models_present,
6,50


Filtered merged shape: (300, 11)


,source,reference,prediction,model,group,input_source,src_norm,ref_norm,pair_key,comet,example_id
0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",baseline,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(""Dad says I'm not cut out for the military, b...",0.924299,6
1,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Papa sagt, ich bin nicht für das Militär gesch...",idiom_only_v1,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(""Dad says I'm not cut out for the military, b...",0.928583,6
2,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",lora_r4_stage2,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(""Dad says I'm not cut out for the military, b...",0.924299,6
3,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",lora_r8_stage2,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(""Dad says I'm not cut out for the military, b...",0.924299,6
4,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",lora_r16_stage2,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(""Dad says I'm not cut out for the military, b...",0.924299,6


## 5. Create a wide-format preview table
This is useful for sanity checking before blinding.

In [12]:
wide_preview = (
    merged.pivot_table(
        index=["example_id", "group", "source", "reference"],
        columns="model",
        values="prediction",
        aggfunc="first"
    )
    .reset_index()
)

# Put the requested model columns in a stable order
preview_cols = ["example_id", "group", "source", "reference"] + [m for m in MODEL_ORDER if m in wide_preview.columns]
wide_preview = wide_preview[preview_cols]

display(wide_preview.head())
print("Wide preview shape:", wide_preview.shape)


model,example_id,group,source,reference,baseline,idiom_only_v1,lora_r4_stage2,lora_r8_stage2,lora_r16_stage2,flan_t5_small_prompt_0shot
0,0,idioms_test,"""Regardless of their cause, inequalities are a...","""Unabhängig von ihrer Ursache sind Ungleichhei...",Unabhängig von ihrer Ursache sind Ungleichheit...,"""Ungleichbehandlungen sind unabhängig von ihre...",Unabhängig von ihrer Ursache sind Ungleichheit...,"""Ungleich, was ihre Ursache betrifft, sind Ung...","""Ungleich, was ihre Ursache betrifft, sind Ung...","""Eigengleichheiten sind eine starken Kräfte fü..."
1,1,idioms_test,A recent public relations disaster left the co...,Die jüngste Katastrophe im Bereich der Öffentl...,Eine kürzliche PR-Katastrophe ließ das Unterne...,Eine jüngste Public-Relations-Katastrophe hat ...,Eine kürzliche PR-Katastrophe ließ das Unterne...,Eine kürzliche Public Relations Desaster hat d...,Eine kürzliche Public Relations Desaster hat d...,Ein jüngsten öffentliche Beziehungensabkommen ...
2,2,idioms_test,"Bend the knee, peasant! Admit that I am your r...","Beuge das Knie, Bauer! Gib zu, dass ich dein r...","Gib zu, dass ich dein rechtmäßiger König bin!","Kniebeugen, Bauer! Gib zu, dass ich dein recht...","Gib zu, dass ich dein rechtmäßiger König bin!","Gib zu, dass ich dein rechtmäßiger König bin!","Gib zu, dass ich dein rechtmäßiger König bin!","Benden die Knee, peasant! Admit, dass ich Ihre..."
3,3,idioms_test,Burning the candle at both ends is a recipe fo...,"Raubbau mit der Gesundheit zu treiben, ist ein...",Das Verbrennen der Kerze an beiden Enden ist e...,Das Verbrennen der Kerze an beiden Enden ist e...,Das Verbrennen der Kerze an beiden Enden ist e...,Das Verbrennen der Kerze an beiden Enden ist e...,Das Verbrennen der Kerze an beiden Enden ist e...,Burning the candle at both ends is a recipe fo...
4,4,idioms_test,But before she could launch her charm offensiv...,Doch bevor sie ihre Charmeoffensive starten ko...,Aber bevor sie ihre Charmeoffensive starten ko...,Aber bevor sie ihre Charmeoffensive starten ko...,Aber bevor sie ihre Charmeoffensive starten ko...,Aber bevor sie ihre Charmeoffensive starten ko...,Aber bevor sie ihre Charmeoffensive starten ko...,"Aber bevor er es er stört, er Herr Roper stört..."


Wide preview shape: (50, 10)


## 6. Build a blinded annotation CSV and a private mapping JSON
This borrows the core idea from the old `20_preds...` notebook, but uses the aligned COMET source-of-truth data.

Important:
- the mapping is shuffled **per example**
- `output_A`, `output_B`, ... can map to different models for different rows
- do not share the mapping JSON with annotators

In [13]:
def build_blinded_annotation_from_wide(
    wide_df: pd.DataFrame,
    out_csv_path: Path,
    out_mapping_json_path: Path,
    *,
    example_id_col: str = "example_id",
    group_col: str = "group",
    source_col: str = "source",
    reference_col: str = "reference",
    model_cols: list[str] | None = None,
    shuffle_per_example: bool = True,
    seed: int = 42,
) -> pd.DataFrame:
    if model_cols is None:
        model_cols = [c for c in wide_df.columns if c not in {example_id_col, group_col, source_col, reference_col}]

    rng = random.Random(seed)
    blinded_rows = []
    mapping = {}

    for _, row in wide_df.iterrows():
        ex_id = str(row[example_id_col])

        model_order = list(model_cols)
        if shuffle_per_example:
            rng.shuffle(model_order)

        output_labels = [chr(ord("A") + i) for i in range(len(model_order))]

        blinded_row = {
            example_id_col: row[example_id_col],
            group_col: row[group_col],
            source_col: row[source_col],
            reference_col: row[reference_col],
        }

        mapping[ex_id] = {}

        for label, model_name in zip(output_labels, model_order):
            blinded_row[f"output_{label}"] = row[model_name]
            mapping[ex_id][f"output_{label}"] = model_name

            # Match prior annotation schema
            blinded_row[f"adequacy_{label}"] = ""
            blinded_row[f"fluency_{label}"] = ""
            blinded_row[f"idiom_correct_{label}"] = ""
            blinded_row[f"over_idiomatization_{label}"] = ""

        blinded_row["best_output"] = ""
        blinded_row["notes"] = ""

        blinded_rows.append(blinded_row)

    blinded_df = pd.DataFrame(blinded_rows)

    out_csv_path.parent.mkdir(parents=True, exist_ok=True)
    blinded_df.to_csv(out_csv_path, index=False)

    with open(out_mapping_json_path, "w", encoding="utf-8") as f:
        json.dump(mapping, f, ensure_ascii=False, indent=2)

    return blinded_df

blinded_df = build_blinded_annotation_from_wide(
    wide_preview,
    OUT_BLINDED_CSV,
    OUT_MAPPING_JSON,
    model_cols=MODEL_ORDER,
    shuffle_per_example=True,
    seed=SEED,
)

print("Saved blinded CSV to:", OUT_BLINDED_CSV)
print("Saved mapping JSON to:", OUT_MAPPING_JSON)
display(blinded_df.head(3))


Saved blinded CSV to: /content/drive/MyDrive/ds266_idiom_mt/qual_preds/preds_6models_annotation_v3_blinded.csv
Saved mapping JSON to: /content/drive/MyDrive/ds266_idiom_mt/qual_preds/preds_6models_annotation_v3_mapping_PRIVATE.json


,example_id,group,source,reference,output_A,adequacy_A,fluency_A,idiom_correct_A,over_idiomatization_A,output_B,...,fluency_E,idiom_correct_E,over_idiomatization_E,output_F,adequacy_F,fluency_F,idiom_correct_F,over_idiomatization_F,best_output,notes
0,0,idioms_test,"""Regardless of their cause, inequalities are a...","""Unabhängig von ihrer Ursache sind Ungleichhei...","""Ungleich, was ihre Ursache betrifft, sind Ung...",,,,,"""Ungleichbehandlungen sind unabhängig von ihre...",...,,,,"""Eigengleichheiten sind eine starken Kräfte fü...",,,,,,
1,1,idioms_test,A recent public relations disaster left the co...,Die jüngste Katastrophe im Bereich der Öffentl...,Eine kürzliche Public Relations Desaster hat d...,,,,,Eine kürzliche PR-Katastrophe ließ das Unterne...,...,,,,Eine jüngste Public-Relations-Katastrophe hat ...,,,,,,
2,2,idioms_test,"Bend the knee, peasant! Admit that I am your r...","Beuge das Knie, Bauer! Gib zu, dass ich dein r...","Gib zu, dass ich dein rechtmäßiger König bin!",,,,,"Kniebeugen, Bauer! Gib zu, dass ich dein recht...",...,,,,"Benden die Knee, peasant! Admit, dass ich Ihre...",,,,,,


## 7. Save helpful companion files
These are for us, not the annotator.

In [14]:
# Save the selected examples list
selected_examples.to_csv(OUT_SELECTED_EXAMPLES_CSV, index=False)
print("Saved selected examples to:", OUT_SELECTED_EXAMPLES_CSV)

# Save the unblinded wide preview for internal use
wide_preview.to_csv(OUT_UNBLINDED_PREVIEW_CSV, index=False)
print("Saved unblinded preview to:", OUT_UNBLINDED_PREVIEW_CSV)

# Quick summary
summary = (
    blinded_df.groupby("group")
    .size()
    .reset_index(name="n_examples")
)

display(summary)

Saved selected examples to: /content/drive/MyDrive/ds266_idiom_mt/qual_preds/preds_6models_annotation_v3_selected_examples.csv
Saved unblinded preview to: /content/drive/MyDrive/ds266_idiom_mt/qual_preds/preds_6models_annotation_v3_unblinded_preview.csv


,group,n_examples
0,idioms_test,25
1,wmt_test,25


## 8. Optional: preview the private mapping for one example
Do **not** share this with the annotator.

In [15]:
example_id = str(blinded_df.loc[0, "example_id"])
with open(OUT_MAPPING_JSON, "r", encoding="utf-8") as f:
    mapping = json.load(f)

print("Example id:", example_id)
print(json.dumps(mapping[example_id], indent=2, ensure_ascii=False))

Example id: 0
{
  "output_A": "lora_r8_stage2",
  "output_B": "idiom_only_v1",
  "output_C": "lora_r4_stage2",
  "output_D": "lora_r16_stage2",
  "output_E": "baseline",
  "output_F": "flan_t5_small_prompt_0shot"
}


## 9. Output files produced by this notebook

Primary annotation package:
- blinded CSV: `preds_6models_annotation_v3_blinded.csv`
- private mapping: `preds_6models_annotation_v3_mapping_PRIVATE.json`

Internal helper files:
- unblinded preview: `preds_6models_annotation_v3_unblinded_preview.csv`
- selected examples: `preds_6models_annotation_v3_selected_examples.csv`

Notes:
- This notebook follows the same annotation schema as the earlier 5-model blinded CSV, but extends it to **6 models** (`output_A` ... `output_F`).
- For WMT examples, the idiom-specific annotation fields can be left blank or treated as N/A by the annotator.
